# Fine-cell conservation in H(div)

Compare locally mixed RT0/P0 Darcy with a primal P1 field followed by constrained energy equilibration. Both fluxes satisfy fine-cell balances; the reconstruction is a minimum-energy constrained operator, distinct from the specific face-moment operator in [Barrenechea et al. (2026)](https://doi.org/10.1137/24M1673073).

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/66c14235a151a8fffbf6dbe7f89d518767b651b983a64da1ba8216ae0bc40248/04_mixed_and_reconstructed_flux-companion.zip"
COMPANION_SHA256 = "66c14235a151a8fffbf6dbe7f89d518767b651b983a64da1ba8216ae0bc40248"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/04_mixed_and_reconstructed_flux.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace
from examples.formulations.application import brinkman as solve_brinkman
from examples.formulations.application import elasticity as solve_elasticity


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

from pymhm.recovery.equilibrated import equilibrate_flux

mesh = TriangleMesh.unit_square(2)
pressure = lambda x: x[:, 0] ** 2 + x[:, 1] ** 2
primal_definition = define_darcy(mesh, source=-4, dirichlet=pressure)
primal_system = assemble(primal_definition.problem)
primal = recover_darcy(
    primal_definition,
    primal_system,
    primal_system.solve(constraints=pressure_constraints(primal_definition, primal_system)),
)
mixed_definition = define_darcy(mesh, source=-4, dirichlet=pressure, formulation="mixed")
mixed_system = assemble(mixed_definition.problem)
mixed = recover_darcy(
    mixed_definition,
    mixed_system,
    mixed_system.solve(constraints=pressure_constraints(mixed_definition, mixed_system)),
)
recovered = equilibrate_flux(primal)
for residual in mixed.fine_conservation_residuals() + recovered.conservation_residuals():
    assert np.max(np.abs(residual)) < 1e-11
print(
    "mixed maximum fine balance",
    max((np.max(np.abs(r)) for r in mixed.fine_conservation_residuals())),
)
print(
    "reconstructed maximum fine balance",
    max((np.max(np.abs(r)) for r in recovered.conservation_residuals())),
)
print("raw primal flux error", primal.flux_l2_error(lambda x: -2 * x))
print("reconstructed flux error", recovered.l2_error(lambda x: -2 * x))


Conservation refers to the assembled source projection. A raw primal gradient generally jumps in its normal component at fine faces and is not a conforming H(div) flux.